Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [5]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [6]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [7]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [8]:
datosNormalizados.shape

(52416, 5)

In [9]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [10]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [11]:
futuros = 1
pasados  = 12

In [12]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [13]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 5)
Dimensiones de Y: (52404, 1)


In [14]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012]]


Se dividen nuevamente los conjuntos de datos

In [15]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 5)
Las dimensiones de testX son:  (10533, 12, 5)
Las dimensiones de valX son:  (5189, 12, 5)


In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [17]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [18]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [19]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [20]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

287/287 - 17s - 61ms/step - ia: 0.1997 - loss: 1.0927 - mae: 0.8645 - rmse: 1.0438 - smape: 1.5746 - val_ia: 0.2207 - val_loss: 0.8478 - val_mae: 0.7655 - val_rmse: 0.9150 - val_smape: 1.8570

Epoch 2/128                                           

287/287 - 4s - 14ms/step - ia: 0.1697 - loss: 1.0112 - mae: 0.8337 - rmse: 1.0042 - smape: 1.6164 - val_ia: 0.2492 - val_loss: 0.7913 - val_mae: 0.7381 - val_rmse: 0.8833 - val_smape: 1.7562

Epoch 3/128                                           

287/287 - 5s - 18ms/step - ia: 0.2833 - loss: 0.8356 - mae: 0.7489 - rmse: 0.9113 - smape: 1.4313 - val_ia: 0.4716 - val_loss: 0.4918 - val_mae: 0.5711 - val_rmse: 0.6943 - val_smape: 1.0538

Epoch 4/128                                           

287/287 - 5s - 18ms/step - ia: 0.5623 - loss: 0.5214 - mae: 0.5787 - rmse: 0.7195 - smape: 1.0024 - val_ia: 0.6724 - val_loss: 0.3068 - val_mae: 0.4375 - val_rmse: 0.5477 - val_smape: 0.7537

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

2293/2293 - 55s - 24ms/step - ia: 0.9173 - loss: 0.0406 - mae: 0.1259 - rmse: 0.1657 - smape: 0.3194 - val_ia: 0.7730 - val_loss: 0.0060 - val_mae: 0.0566 - val_rmse: 0.0691 - val_smape: 0.1497

Epoch 2/128                                                                         

2293/2293 - 48s - 21ms/step - ia: 0.9698 - loss: 0.0046 - mae: 0.0470 - rmse: 0.0621 - smape: 0.1557 - val_ia: 0.8188 - val_loss: 0.0034 - val_mae: 0.0421 - val_rmse: 0.0527 - val_smape: 0.1371

Epoch 3/128                                                                         

2293/2293 - 48s - 21ms/step - ia: 0.9730 - loss: 0.0039 - mae: 0.0423 - rmse: 0.0563 - smape: 0.1422 - val_ia: 0.8392 - val_loss: 0.0027 - val_mae: 0.0373 - val_rmse: 0.0478 - val_smape: 0.1269

Epoch 4/128                                                                         

2293/2293 - 47s - 20ms/step - ia: 0.9741 - loss: 0.0037 - mae: 0.0405 -

...

Epoch 1/128                                                                           

2293/2293 - 28s - 12ms/step - ia: 0.6800 - loss: 0.3479 - mae: 0.4437 - rmse: 0.5452 - smape: 0.8356 - val_ia: 0.4039 - val_loss: 0.1550 - val_mae: 0.3085 - val_rmse: 0.3317 - val_smape: 0.6487

Epoch 2/128                                                                           

2293/2293 - 27s - 12ms/step - ia: 0.8272 - loss: 0.1166 - mae: 0.2663 - rmse: 0.3342 - smape: 0.5623 - val_ia: 0.4825 - val_loss: 0.0913 - val_mae: 0.2375 - val_rmse: 0.2562 - val_smape: 0.5565

Epoch 3/128                                                                           

2293/2293 - 25s - 11ms/step - ia: 0.8667 - loss: 0.0715 - mae: 0.2064 - rmse: 0.2612 - smape: 0.4648 - val_ia: 0.5649 - val_loss: 0.0458 - val_mae: 0.1668 - val_rmse: 0.1850 - val_smape: 0.4474

Epoch 4/128                                                                           

2293/2293 - 24s - 11ms/step - ia: 0.8883 - loss: 0.0506 - mae: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                             

4586/4586 - 44s - 10ms/step - ia: 0.2316 - loss: 0.9685 - mae: 0.8148 - rmse: 0.9645 - smape: 1.6991 - val_ia: 0.1426 - val_loss: 0.8622 - val_mae: 0.7716 - val_rmse: 0.7886 - val_smape: 1.7005

Epoch 2/128                                                                             

4586/4586 - 32s - 7ms/step - ia: 0.2478 - loss: 0.9307 - mae: 0.7977 - rmse: 0.9451 - smape: 1.6674 - val_ia: 0.1445 - val_loss: 0.8270 - val_mae: 0.7558 - val_rmse: 0.7726 - val_smape: 1.6944

Epoch 3/128                                                                             

4586/4586 - 31s - 7ms/step - ia: 0.2662 - loss: 0.8925 - mae: 0.7799 - rmse: 0.9254 - smape: 1.6329 - val_ia: 0.1464 - val_loss: 0.7894 - val_mae: 0.7387 - val_rmse: 0.7551 - val_smape: 1.6795

Epoch 4/128                                                                             

4586/4586 - 41s - 9ms/step - ia: 0.2861 - loss: 0.8552 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                             

287/287 - 9s - 32ms/step - ia: 0.6582 - loss: 0.3611 - mae: 0.4566 - rmse: 0.5632 - smape: 0.8871 - val_ia: 0.7811 - val_loss: 0.1397 - val_mae: 0.3028 - val_rmse: 0.3697 - val_smape: 0.6705

Epoch 2/128                                                                             

287/287 - 4s - 14ms/step - ia: 0.8283 - loss: 0.1260 - mae: 0.2777 - rmse: 0.3536 - smape: 0.6009 - val_ia: 0.8064 - val_loss: 0.1109 - val_mae: 0.2709 - val_rmse: 0.3281 - val_smape: 0.6268

Epoch 3/128                                                                             

287/287 - 5s - 17ms/step - ia: 0.8558 - loss: 0.0895 - mae: 0.2342 - rmse: 0.2979 - smape: 0.5332 - val_ia: 0.8310 - val_loss: 0.0889 - val_mae: 0.2360 - val_rmse: 0.2845 - val_smape: 0.5657

Epoch 4/128                                                                             

287/287 - 8s - 28ms/step - ia: 0.8764 - loss: 0.0670 - mae: 0.20

In [21]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
